In [1]:
import os
from dotenv import load_dotenv
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap


load_dotenv(override=True)

True

In [4]:
doc_loader = TextLoader("../data/text_files/artificial_intelligence.txt")

raw_docs = doc_loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(raw_docs)

chunks

[Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='What is artificial intelligence (AI)?'),
 Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='Artificial intelligence (AI) is a set of technologies that empowers computers to learn, reason, and perform a variety of advanced tasks in ways that used to require human intelligence, such as'),
 Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='such as understanding language, analyzing data, and even providing helpful suggestions. It’s a transformational technology that can bring meaningful and positive change to people and societies and'),
 Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='and societies and the world.'),
 Document(metadata={'source': '../data/text_files/artificial_intelligence.txt'}, page_content='It encompasses many different disciplines, including com

In [5]:
embedding_model = HuggingFaceEmbeddings(model_name = os.getenv("HUGGING_FACE_EMBEDDING_MODEL"))

vector_store = FAISS.from_documents(chunks, embedding_model)

retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k":5})

retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12554.51it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x126341fd0>, search_type='mmr', search_kwargs={'k': 5})

In [6]:
llm = init_chat_model(os.getenv("OLLAMA_CHAT_MODEL"))

In [7]:
#Query expansion

query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrival by adding relevant synonyms, technical terms, and useful context.

Qriginal query: "{query}"

Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()

query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrival by adding relevant synonyms, technical terms, and useful context.\n\nQriginal query: "{query}"\n\nExpanded query:\n')
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')
| StrOutputParser()

In [8]:
query_expansion_chain.invoke({"query":"AI"})

'This is an extremely broad query, so the expansion needs to cover multiple dimensions: **the definition**, **the underlying technology**, **the methodologies**, and **the applications**.\n\nBelow are several expanded versions, depending on whether you are looking for a general overview, a highly technical dive, or practical applications.\n\n***\n\n### 💡 Recommended Universal Expansion (The Best Starting Point)\n\nThis comprehensive version uses multiple keywords connected by implied boolean operators (`OR` or `AND`) to maximize retrieval without assuming a specific focus.\n\n**Expanded query:**\n`"Artificial Intelligence" OR "AI systems" OR "cognitive computing" OR "machine intelligence" AND (machine learning OR deep learning OR neural networks OR natural language processing OR computer vision OR generative AI) AND (algorithms OR predictive analytics OR AI models OR large language models OR LLMs)`\n\n***\n\n### 💻 Alternative Expansions (Choose based on your intent)\n\n#### 1. For a **

In [9]:
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

In [10]:
document_chain = create_stuff_documents_chain(llm=llm, prompt=answer_prompt)



In [11]:
rag_pipeline = (
    RunnableMap(
        {
            "input": lambda x: x["input"],
            "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query":x["input"]}))
        }
    )
    | document_chain
)

In [12]:
query = {"input": "What are the modules of AI?"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query:**

"Comprehensive overview of Artificial Intelligence (AI) core components, foundational modules, and sub-fields. Specifically, what are the primary pillars of AI technology, including **Machine Learning (ML)**, **Deep Learning (DL)**, **Natural Language Processing (NLP)**, **Computer Vision (CV)**, **Robotics**, and **Expert Systems**? Please detail the relationship and hierarchical structure among these domains. We are looking for information regarding AI architecture, key algorithms (e.g., neural networks, transformers), and the operational modules that constitute modern AI systems."

***

### 💡 Breakdown of Improvements:

**1. Added Synonyms & Scope:**
* **Original:** "What are the modules of AI?"
* **Improvement:** Added "core components," "foundational modules," "pillars," and "overview." (This helps the search engine understand that "modules" could mean structure, sub-fields, or components.)

**2. Added Specific Technical Terms (Mandatory Keywords):**
* Explici